# MobileNetV2 + TSM — Frame-Diversity Held-Out Evaluation
## RWF-2000 + UCF-Crime Fighting + Bus Violence

Notebook ini **TEST ONLY / EVALUATION ONLY**.

Tidak ada:
- training ulang;
- optimizer;
- scheduler;
- backpropagation;
- fine-tuning;
- model selection menggunakan held-out dataset.

Basis protokol mengikuti:
`Zhang2022_MobileNetTSM_FrameDiversity_123Seed.ipynb`.

Arsitektur dan frame-diversity protocol yang dipertahankan:

```text
shared 16-frame cache
→ fixed T=8 evaluation grid [1,3,5,7,9,11,13,15]
→ pilih K distinct temporal positions pada grid T=8
→ K ∈ {1,2,4,8}
→ equal-weight chronological repetition sampai sequence kembali T=8
→ MobileNetV2 + Zhang Method-2 TSM
→ per-segment logits
→ average-logit consensus
→ clip prediction
```

Untuk:
- `K=1,2,4` → 10 temporal-sampling repeats;
- `K=8` → satu deterministic reference.

> **K=1 bukan native-T=1 checkpoint.**
>
> Checkpoint tetap **MobileNetV2+TSM T=8** dan interface model tetap T=8.
> Yang dikurangi hanya jumlah distinct temporal observations pada sequence
> yang tetap memiliki 8 posisi.

Berbeda dengan MobileNetV2 tanpa TSM, pada model ini pengulangan frame juga
mengubah pola adjacency yang dilihat TSM. Karena itu eksperimen ini mengukur
sensitivitas terhadap temporal diversity/redundancy pada fixed T=8, bukan
isolasi kausal murni jumlah frame.

Held-out dataset dievaluasi **terpisah**:
1. RWF-2000
2. UCF-Crime Fighting
3. Bus Violence

Tidak ada pooling RWF + UCF + Bus.

## Mengganti seed

Setelah path dataset benar, **cukup ubah satu baris**:

```python
RUN_SEED = 0
```

menjadi `42` atau `123`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import gc
import json
import random
import platform
import warnings
import zlib

from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
)

import torchvision

from torchvision.models import (
    mobilenet_v2,
)

from torchvision.models.mobilenetv2 import (
    InvertedResidual,
)

from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

print(
    "Python      :",
    platform.python_version(),
)

print(
    "PyTorch     :",
    torch.__version__,
)

print(
    "torchvision :",
    torchvision.__version__,
)

print(
    "CUDA        :",
    torch.cuda.is_available(),
)

if torch.cuda.is_available():
    print(
        "GPU         :",
        torch.cuda.get_device_name(
            0
        ),
    )

Python      : 3.13.15
PyTorch     : 2.11.0+cu128
torchvision : 0.26.0+cu128
CUDA        : True
GPU         : NVIDIA A100-SXM4-40GB


## 1. Konfigurasi

### Satu-satunya variabel yang diubah antar-seed

```python
RUN_SEED = 0
```

Valid:

```python
RUN_SEED ∈ {0,42,123}
```

Jangan ubah antar-seed:

```python
NUM_SEGMENTS = 8
SHIFT_DIV = 8
TEST_SEGMENT_INDICES = [1,3,5,7,9,11,13,15]
K_VALUES = [1,2,4,8]
NUM_RANDOM_REPEATS = 10
SAMPLING_BASE_SEED = 20260810
```

`RUN_SEED` **tidak masuk** ke generator temporal sampling. Dengan demikian,
checkpoint seed 0/42/123 menerima temporal selections yang sama untuk
dataset, sample, K, dan repeat yang sama.

In [ ]:
# ============================================================
# SATU-SATUNYA VARIABEL YANG DIUBAH ANTAR-SEED
# ============================================================
RUN_SEED = 0

VALID_SEEDS = (
    0,
    42,
    123,
)

if RUN_SEED not in VALID_SEEDS:
    raise ValueError(
        f"RUN_SEED harus salah satu dari {VALID_SEEDS}; "
        f"mendapat {RUN_SEED}."
    )

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

# ============================================================
# EXACT T=8 MODEL PROTOCOL
# ============================================================
MODEL_VARIANT = "mobilenet_tsm"

USE_TSM = True

NUM_CLASSES = 2

CACHE_SEQUENCE_LENGTH = 16

RAW_HEIGHT = 64
RAW_WIDTH = 64

NUM_SEGMENTS = 8

BACKBONE_INPUT_SIZE = 224

SHIFT_DIV = 8

TEST_SEGMENT_INDICES = np.asarray(
    [
        1,
        3,
        5,
        7,
        9,
        11,
        13,
        15,
    ],
    dtype=np.int64,
)

K_VALUES = [
    1,
    2,
    4,
    8,
]

NUM_RANDOM_REPEATS = 10

# Must remain identical across checkpoint seeds.
SAMPLING_BASE_SEED = 20260810

BATCH_SIZE = 8

NUM_WORKERS = 2

PREFER_LEGACY_CACHE = True

LEGACY_CACHE_COLOR_ORDER = "BGR"

AUTO_SCALE_LEGACY_CACHE = True

ALLOW_BUILD_CACHE = True

# TSM training/source notebook explicitly expects 10 blocks.
EXPECTED_TSM_BLOCKS = 10

# K=8 frame-diversity route must reproduce direct T=8 route.
K8_PROBABILITY_ATOL = 1e-6
K8_METRIC_ATOL = 1e-8

# ============================================================
# MODEL PATHS
# ============================================================
MODELS_ROOT = Path(
    "/content/drive/MyDrive/IVS/Models"
)

COMPARATOR_ROOT = (
    MODELS_ROOT
    / "Comparators"
)

EFFTEMPNET_ROOT = (
    MODELS_ROOT
    / "Model"
)

CHECKPOINT_PATH_OVERRIDE = None


def resolve_checkpoint(
    seed: int,
) -> Path:
    if CHECKPOINT_PATH_OVERRIDE is not None:
        path = Path(
            CHECKPOINT_PATH_OVERRIDE
        )

        if not path.exists():
            raise FileNotFoundError(
                f"CHECKPOINT_PATH_OVERRIDE tidak ditemukan: {path}"
            )

        return path

    # Primary naming from uploaded Frame-Diversity source:
    # mobilenet_tsm_SeedX_v1/mobilenet_tsm/seed_X/best.pt
    #
    # Also support older Zhang2022_ prefixed naming.
    candidates = [
        (
            COMPARATOR_ROOT
            / f"mobilenet_tsm_Seed{seed}_v1"
            / "mobilenet_tsm"
            / f"seed_{seed}"
            / "best.pt"
        ),
        (
            COMPARATOR_ROOT
            / f"mobilenet_tsm_Seed{seed}_v1"
            / "mobilenet_tsm"
            / f"seed_{seed}"
            / "best.pt"
        ),
        (
            COMPARATOR_ROOT
            / f"mobilenet_tsm_Seed{seed}"
            / "mobilenet_tsm"
            / f"seed_{seed}"
            / "best.pt"
        ),
    ]

    matches = [
        path
        for path in candidates
        if path.exists()
    ]

    matches = list(
        dict.fromkeys(
            matches
        )
    )

    if len(
        matches
    ) == 1:
        return matches[
            0
        ]

    if len(
        matches
    ) > 1:
        raise RuntimeError(
            "Ditemukan beberapa checkpoint MobileNetV2+TSM T=8. "
            "Set CHECKPOINT_PATH_OVERRIDE secara eksplisit:\n"
            + "\n".join(
                str(
                    path
                )
                for path
                in matches
            )
        )

    # Safe fallback:
    # exact mobilenet_tsm/seed_X/best.pt only,
    # excluding native-T variants and unrelated TSM families.
    recursive = []

    for path in COMPARATOR_ROOT.glob(
        f"**/mobilenet_tsm/seed_{seed}/best.pt"
    ):
        low = str(
            path
        ).lower()

        if (
            "native_t" in low
            or "nativet" in low
        ):
            continue

        if "wang2024" in low:
            continue

        recursive.append(
            path
        )

    recursive = sorted(
        dict.fromkeys(
            recursive
        )
    )

    if len(
        recursive
    ) == 1:
        return recursive[
            0
        ]

    if len(
        recursive
    ) == 0:
        raise FileNotFoundError(
            "Checkpoint MobileNetV2+TSM T=8 tidak ditemukan.\n"
            f"Seed={seed}\n"
            f"Search root={COMPARATOR_ROOT}\n"
            "Jika struktur folder berbeda, isi CHECKPOINT_PATH_OVERRIDE."
        )

    raise RuntimeError(
        "Ditemukan beberapa checkpoint MobileNetV2+TSM T=8 yang ambigu. "
        "Set CHECKPOINT_PATH_OVERRIDE:\n"
        + "\n".join(
            str(
                path
            )
            for path
            in recursive
        )
    )


CHECKPOINT_PATH = resolve_checkpoint(
    RUN_SEED
)

# ============================================================
# HELD-OUT DATASET PATHS
# ============================================================
BENCHMARK_ROOT = Path(
    "/content/drive/MyDrive/Benchmark"
)

EXISTING_CACHE_DIR = (
    EFFTEMPNET_ROOT
    / "new_saved_dataset"
)

HELDOUT_CACHE_DIR = (
    EFFTEMPNET_ROOT
    / "heldout_cache_rgb64_v2"
)

HELDOUT_SPECS = {
    "RWF": {
        "raw_dir": (
            BENCHMARK_ROOT
            / "RWF"
        ),
        "cache_name": "RWF",
        "cache_dir": EXISTING_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },

    "UCF": {
        "raw_dir": (
            BENCHMARK_ROOT
            / "UCF"
        ),
        "cache_name": "UCF",
        "cache_dir": HELDOUT_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },

    "BV": {
        "raw_dir": (
            BENCHMARK_ROOT
            / "BV"
        ),
        "cache_name": "BV",
        "cache_dir": HELDOUT_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },
}

NONVIOLENCE_ALIASES = (
    "NonViolence",
    "Non-Violence",
    "Non_Violence",
    "Nonviolent",
    "NonViolent",
    "NonFight",
    "Non-Fight",
    "Non_Fight",
    "Normal",
    "NormalVideos",
    "Normal_Videos",
)

VIOLENCE_ALIASES = (
    "Violence",
    "Violent",
    "Fight",
    "Fighting",
    "FightVideos",
    "ViolentVideos",
    "Abnormal",
)

EVAL_OUTPUT_DIR = (
    EFFTEMPNET_ROOT
    / "HeldOut_Evaluation"
    / "MobileNetV2_TSM_FrameDiversity"
    / f"seed_{RUN_SEED}"
)

EVAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HELDOUT_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

for k_unique in K_VALUES:
    if (
        NUM_SEGMENTS
        % k_unique
        != 0
    ):
        raise ValueError(
            f"K={k_unique} tidak membagi T={NUM_SEGMENTS}."
        )

if (
    TEST_SEGMENT_INDICES.tolist()
    != [
        1,
        3,
        5,
        7,
        9,
        11,
        13,
        15,
    ]
):
    raise RuntimeError(
        "Fixed T=8 evaluation indices berubah."
    )

print(
    "RUN_SEED      :",
    RUN_SEED,
)

print(
    "DEVICE        :",
    DEVICE,
)

print(
    "CHECKPOINT    :",
    CHECKPOINT_PATH,
)

print(
    "USE_TSM       :",
    USE_TSM,
)

print(
    "SHIFT_DIV     :",
    SHIFT_DIV,
)

print(
    "T=8 indices   :",
    TEST_SEGMENT_INDICES.tolist(),
)

print(
    "K values      :",
    K_VALUES,
)

print(
    "Repeats       :",
    NUM_RANDOM_REPEATS,
)

print(
    "Sampling seed :",
    SAMPLING_BASE_SEED,
)

print(
    "OUTPUT        :",
    EVAL_OUTPUT_DIR,
)

RUN_SEED      : 0
DEVICE        : cuda
CHECKPOINT    : /content/drive/MyDrive/IVS/ViolenceDetection/Models/Comparators/mobilenet_tsm_Seed0_v1/mobilenet_tsm/seed_0/best.pt
USE_TSM       : True
SHIFT_DIV     : 8
T=8 indices   : [1, 3, 5, 7, 9, 11, 13, 15]
K values      : [1, 2, 4, 8]
Repeats       : 10
Sampling seed : 20260810
OUTPUT        : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/MobileNetV2_TSM_FrameDiversity/seed_0


In [ ]:
def set_eval_seed(
    seed: int,
):
    os.environ[
        "PYTHONHASHSEED"
    ] = str(
        seed
    )

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(
    worker_id: int,
):
    worker_seed = (
        torch.initial_seed()
        % (2**32)
    )

    np.random.seed(
        worker_seed
    )

    random.seed(
        worker_seed
    )


def safe_torch_load(
    path: Path,
    map_location,
):
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=False,
        )

    except TypeError:
        return torch.load(
            path,
            map_location=map_location,
        )


def assert_finite_tensor(
    name: str,
    tensor: torch.Tensor,
):
    if not torch.isfinite(
        tensor
    ).all():
        raise RuntimeError(
            f"{name} mengandung NaN/Inf."
        )


set_eval_seed(
    RUN_SEED
)

## 2. Exact MobileNetV2 + TSM architecture

Sama dengan notebook source:

- TSM dipasang pada residual `InvertedResidual` yang eligible;
- diletakkan setelah expansion layer;
- `SHIFT_DIV = 8`;
- expected TSM blocks = 10;
- model input tetap T=8;
- output clip menggunakan mean-logit consensus.

Pada Frame Diversity K<8, TSM tetap aktif dan memproses sequence T=8 hasil
equal-weight repetition.

In [ ]:
class TemporalShift(
    nn.Module
):
    def __init__(
        self,
        n_segment: int = NUM_SEGMENTS,
        n_div: int = SHIFT_DIV,
    ):
        super().__init__()

        self.n_segment = int(
            n_segment
        )

        self.n_div = int(
            n_div
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        if x.ndim != 4:
            raise ValueError(
                f"TemporalShift expects [B*T,C,H,W], "
                f"got {tuple(x.shape)}"
            )

        (
            nt,
            channels,
            height,
            width,
        ) = x.shape

        if (
            nt
            % self.n_segment
            != 0
        ):
            raise ValueError(
                f"B*T={nt} tidak habis dibagi "
                f"T={self.n_segment}."
            )

        batch_size = (
            nt
            // self.n_segment
        )

        x = x.view(
            batch_size,
            self.n_segment,
            channels,
            height,
            width,
        )

        fold = (
            channels
            // self.n_div
        )

        if fold == 0:
            return x.view(
                nt,
                channels,
                height,
                width,
            )

        out = torch.zeros_like(
            x
        )

        # First channel group shifts toward previous temporal position.
        out[
            :,
            :-1,
            :fold,
        ] = x[
            :,
            1:,
            :fold,
        ]

        # Second channel group shifts toward next temporal position.
        out[
            :,
            1:,
            fold:2 * fold,
        ] = x[
            :,
            :-1,
            fold:2 * fold,
        ]

        # Remaining channels are unchanged.
        out[
            :,
            :,
            2 * fold:,
        ] = x[
            :,
            :,
            2 * fold:,
        ]

        return out.view(
            nt,
            channels,
            height,
            width,
        )


def insert_zhang_method2_tsm(
    model: nn.Module,
    n_segment: int = NUM_SEGMENTS,
    n_div: int = SHIFT_DIV,
) -> int:
    inserted = 0

    for block in model.features:
        if not isinstance(
            block,
            InvertedResidual,
        ):
            continue

        if not block.use_res_connect:
            continue

        if len(
            block.conv
        ) < 4:
            continue

        expansion = (
            block.conv[
                0
            ]
        )

        block.conv[
            0
        ] = nn.Sequential(
            expansion,
            TemporalShift(
                n_segment=n_segment,
                n_div=n_div,
            ),
        )

        inserted += 1

    return inserted


class VideoMobileNetTSM(
    nn.Module
):
    def __init__(
        self,
        num_classes: int = NUM_CLASSES,
    ):
        super().__init__()

        self.use_tsm = True

        self.num_segments = NUM_SEGMENTS

        self.shift_div = SHIFT_DIV

        self.base_model = mobilenet_v2(
            weights=None
        )

        in_features = int(
            self.base_model
            .classifier[
                1
            ]
            .in_features
        )

        self.base_model.classifier[
            1
        ] = nn.Linear(
            in_features,
            num_classes,
        )

        self.tsm_blocks = (
            insert_zhang_method2_tsm(
                self.base_model,
                n_segment=NUM_SEGMENTS,
                n_div=SHIFT_DIV,
            )
        )

    def forward(
        self,
        clip: torch.Tensor,
    ) -> torch.Tensor:
        if clip.ndim != 5:
            raise ValueError(
                f"Expected [B,T,C,H,W], "
                f"got {tuple(clip.shape)}"
            )

        (
            batch_size,
            seq_len,
            channels,
            height,
            width,
        ) = clip.shape

        if (
            seq_len
            != self.num_segments
        ):
            raise ValueError(
                f"Expected {self.num_segments} segments, "
                f"got {seq_len}"
            )

        if channels != 3:
            raise ValueError(
                f"Expected RGB channels=3, got {channels}"
            )

        x = clip.reshape(
            batch_size
            * seq_len,
            channels,
            height,
            width,
        )

        x = F.interpolate(
            x,
            size=(
                BACKBONE_INPUT_SIZE,
                BACKBONE_INPUT_SIZE,
            ),
            mode="bilinear",
            align_corners=False,
        )

        segment_logits = (
            self.base_model(
                x
            )
        )

        segment_logits = (
            segment_logits
            .view(
                batch_size,
                seq_len,
                NUM_CLASSES,
            )
        )

        return (
            segment_logits
            .mean(
                dim=1
            )
        )


# ============================================================
# STRICT CHECKPOINT LOAD
# ============================================================
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint tidak ditemukan: {CHECKPOINT_PATH}"
    )

checkpoint = safe_torch_load(
    CHECKPOINT_PATH,
    map_location="cpu",
)

if not isinstance(
    checkpoint,
    dict,
):
    raise TypeError(
        "Checkpoint bukan dictionary."
    )

if (
    "model_state_dict"
    not in checkpoint
):
    raise KeyError(
        "Checkpoint tidak mempunyai model_state_dict."
    )

checkpoint_config = checkpoint.get(
    "config",
    {},
)

if "seed" in checkpoint_config:
    if int(
        checkpoint_config[
            "seed"
        ]
    ) != RUN_SEED:
        raise RuntimeError(
            "Checkpoint seed mismatch."
        )

if "use_tsm" in checkpoint_config:
    if not bool(
        checkpoint_config[
            "use_tsm"
        ]
    ):
        raise RuntimeError(
            "Checkpoint metadata menyatakan use_tsm=False."
        )

if "num_segments" in checkpoint_config:
    if int(
        checkpoint_config[
            "num_segments"
        ]
    ) != NUM_SEGMENTS:
        raise RuntimeError(
            "Checkpoint bukan T=8."
        )

if "shift_div" in checkpoint_config:
    if int(
        checkpoint_config[
            "shift_div"
        ]
    ) != SHIFT_DIV:
        raise RuntimeError(
            "Checkpoint shift_div mismatch."
        )

model = VideoMobileNetTSM().to(
    DEVICE
)

if (
    model.tsm_blocks
    != EXPECTED_TSM_BLOCKS
):
    raise RuntimeError(
        f"Expected {EXPECTED_TSM_BLOCKS} TSM blocks, "
        f"got {model.tsm_blocks}."
    )

load_result = model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ],
    strict=True,
)

model.eval()

for parameter in model.parameters():
    parameter.requires_grad_(
        False
    )

if not model.use_tsm:
    raise RuntimeError(
        "TSM harus aktif."
    )

runtime_shifts = [
    module
    for module
    in model.modules()
    if isinstance(
        module,
        TemporalShift,
    )
]

if (
    len(
        runtime_shifts
    )
    != EXPECTED_TSM_BLOCKS
):
    raise RuntimeError(
        f"Expected {EXPECTED_TSM_BLOCKS} TemporalShift modules, "
        f"got {len(runtime_shifts)}."
    )

for shift_module in runtime_shifts:
    if (
        shift_module.n_segment
        != NUM_SEGMENTS
    ):
        raise RuntimeError(
            "TemporalShift n_segment mismatch."
        )

    if (
        shift_module.n_div
        != SHIFT_DIV
    ):
        raise RuntimeError(
            "TemporalShift shift_div mismatch."
        )

with torch.inference_mode():
    dummy = torch.zeros(
        2,
        NUM_SEGMENTS,
        3,
        RAW_HEIGHT,
        RAW_WIDTH,
        device=DEVICE,
    )

    dummy_logits = model(
        dummy
    )

if (
    tuple(
        dummy_logits.shape
    )
    != (
        2,
        NUM_CLASSES,
    )
):
    raise RuntimeError(
        f"Unexpected output shape: "
        f"{tuple(dummy_logits.shape)}"
    )

assert_finite_tensor(
    "dummy_logits",
    dummy_logits,
)

total_params = sum(
    parameter.numel()
    for parameter
    in model.parameters()
)

print(
    "✅ Strict MobileNetV2+TSM checkpoint/model audit PASS"
)

print(
    "Epoch      :",
    checkpoint.get(
        "epoch",
        "unknown",
    ),
)

print(
    "Best score :",
    checkpoint.get(
        "best_score",
        "unknown",
    ),
)

print(
    "TSM blocks :",
    model.tsm_blocks,
)

print(
    "Segments   :",
    model.num_segments,
)

print(
    "Shift div  :",
    model.shift_div,
)

print(
    "Parameters :",
    f"{total_params:,}",
)

print(
    "Load result:",
    load_result,
)

del (
    dummy,
    dummy_logits,
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

✅ Strict MobileNetV2+TSM checkpoint/model audit PASS
Epoch      : 63
Best score : 0.9504283965728274
TSM blocks : 10
Segments   : 8
Shift div  : 8
Parameters : 2,226,434
Load result: <All keys matched successfully>


## 3. Resolve / build shared held-out 16-frame cache

RWF menggunakan cache eksperimen yang sudah ada jika tersedia.

UCF dan Bus Violence memakai shared held-out RGB-v2 cache yang sama dengan
evaluator sebelumnya.

Raw-video cache **tidak bergantung pada checkpoint seed**.

In [ ]:
VIDEO_EXTENSIONS = {
    ".mp4", ".avi", ".mov", ".mkv",
    ".mpeg", ".mpg", ".webm"
}

def direct_subdirs(root: Path) -> Dict[str, Path]:
    if not root.exists():
        return {}
    return {
        p.name: p
        for p in root.iterdir()
        if p.is_dir()
    }

def resolve_one_class_dir(
    root: Path,
    explicit_name: Optional[str],
    aliases: Tuple[str, ...],
    canonical_name: str,
) -> Path:
    if explicit_name is not None:
        candidate = root / explicit_name
        if not candidate.exists():
            raise FileNotFoundError(
                f"Folder {canonical_name} tidak ditemukan: {candidate}"
            )
        return candidate

    subdirs = direct_subdirs(root)
    lower_map = {
        name.lower(): path
        for name, path in subdirs.items()
    }

    matches = []
    for alias in aliases:
        candidate = lower_map.get(alias.lower())
        if candidate is not None and candidate not in matches:
            matches.append(candidate)

    if len(matches) != 1:
        raise RuntimeError(
            f"Tidak dapat menentukan folder kelas {canonical_name} "
            f"secara aman pada {root}.\n"
            f"Matches: {[p.name for p in matches]}\n"
            f"Available: {sorted(subdirs.keys())}\n"
            "Isi nonviolence_dir/violence_dir di HELDOUT_SPECS."
        )

    return matches[0]

def list_videos_for_spec(
    raw_dir: Path,
    nonviolence_dir: Optional[str],
    violence_dir: Optional[str],
) -> List[Tuple[Path, int]]:
    if not raw_dir.exists():
        raise FileNotFoundError(
            f"Raw held-out directory tidak ditemukan: {raw_dir}"
        )

    class_dirs = {
        0: resolve_one_class_dir(
            raw_dir,
            nonviolence_dir,
            NONVIOLENCE_ALIASES,
            "NonViolence",
        ),
        1: resolve_one_class_dir(
            raw_dir,
            violence_dir,
            VIOLENCE_ALIASES,
            "Violence",
        ),
    }

    records = []

    for label, class_dir in class_dirs.items():
        videos = sorted(
            p for p in class_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
        )

        if not videos:
            raise RuntimeError(
                f"Tidak ada video pada folder kelas: {class_dir}"
            )

        print(
            f"{CLASS_NAMES[label]:12s}: {len(videos):5d} clips | {class_dir}"
        )

        records.extend(
            (p, label)
            for p in videos
        )

    labels = np.asarray(
        [label for _, label in records],
        dtype=np.int64,
    )

    if set(np.unique(labels).tolist()) != {0, 1}:
        raise RuntimeError(
            "Held-out dataset harus mempunyai kedua kelas: "
            "NonViolence dan Violence."
        )

    return records

def extract_uniform_rgb_frames(
    video_path: Path,
    sequence_length: int = CACHE_SEQUENCE_LENGTH,
    height: int = RAW_HEIGHT,
    width: int = RAW_WIDTH,
) -> Optional[np.ndarray]:
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        return None

    frame_count = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    if frame_count <= 0:
        cap.release()
        return None

    frame_positions = np.linspace(
        0,
        max(frame_count - 1, 0),
        sequence_length,
    ).astype(int)

    frames = []

    for pos in frame_positions:
        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(pos),
        )

        ok, frame_bgr = cap.read()

        if not ok:
            break

        frame_rgb = cv2.cvtColor(
            frame_bgr,
            cv2.COLOR_BGR2RGB,
        )

        frame_rgb = cv2.resize(
            frame_rgb,
            (width, height),
            interpolation=cv2.INTER_LINEAR,
        )

        frames.append(
            frame_rgb.astype(np.float32) / 255.0
        )

    cap.release()

    if not frames:
        return None

    while len(frames) < sequence_length:
        frames.append(frames[-1].copy())

    return np.stack(
        frames[:sequence_length],
        axis=0,
    ).astype(np.float32)

def build_rgb_cache(
    raw_dir: Path,
    cache_dir: Path,
    cache_name: str,
    nonviolence_dir: Optional[str],
    violence_dir: Optional[str],
):
    cache_dir.mkdir(parents=True, exist_ok=True)

    feature_path = cache_dir / f"{cache_name}_rgb64_v2_features.npy"
    label_path = cache_dir / f"{cache_name}_rgb64_v2_labels.npy"
    paths_path = cache_dir / f"{cache_name}_rgb64_v2_paths.json"

    records = list_videos_for_spec(
        raw_dir,
        nonviolence_dir,
        violence_dir,
    )

    # Gunakan temporary memmap agar tidak menyimpan semua clip di RAM.
    n_requested = len(records)

    temp_path = cache_dir / f".{cache_name}_tmp_features.npy"

    if temp_path.exists():
        temp_path.unlink()

    temp = np.lib.format.open_memmap(
        temp_path,
        mode="w+",
        dtype=np.float32,
        shape=(
            n_requested,
            CACHE_SEQUENCE_LENGTH,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        ),
    )

    labels_buffer = np.empty(
        n_requested,
        dtype=np.int64,
    )

    valid_paths = []
    failed_paths = []
    valid_count = 0

    for video_path, label in tqdm(
        records,
        desc=f"Build cache {cache_name}",
        unit="video",
    ):
        clip = extract_uniform_rgb_frames(video_path)

        if clip is None:
            failed_paths.append(str(video_path))
            continue

        temp[valid_count] = clip
        labels_buffer[valid_count] = int(label)
        valid_paths.append(str(video_path))
        valid_count += 1

    temp.flush()
    del temp

    if valid_count == 0:
        if temp_path.exists():
            temp_path.unlink()
        raise RuntimeError(
            f"Semua video gagal didekode untuk {cache_name}."
        )

    # Compact hanya sample valid ke final .npy.
    temp_read = np.load(
        temp_path,
        mmap_mode="r",
    )

    final = np.lib.format.open_memmap(
        feature_path,
        mode="w+",
        dtype=np.float32,
        shape=(
            valid_count,
            CACHE_SEQUENCE_LENGTH,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        ),
    )

    chunk_size = 16

    for start in range(0, valid_count, chunk_size):
        end = min(start + chunk_size, valid_count)
        final[start:end] = temp_read[start:end]

    final.flush()
    del final, temp_read

    np.save(
        label_path,
        labels_buffer[:valid_count],
    )

    with open(
        paths_path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            {
                "paths": valid_paths,
                "failed": failed_paths,
                "n_requested": n_requested,
                "n_valid": valid_count,
                "color_order": "RGB",
                "sequence_length": CACHE_SEQUENCE_LENGTH,
                "height": RAW_HEIGHT,
                "width": RAW_WIDTH,
            },
            f,
            indent=2,
        )

    if temp_path.exists():
        temp_path.unlink()

    print(
        f"Saved {feature_path} | "
        f"shape=({valid_count},{CACHE_SEQUENCE_LENGTH},{RAW_HEIGHT},{RAW_WIDTH},3)"
    )
    print("Failed videos:", len(failed_paths))

    return {
        "feature_path": feature_path,
        "label_path": label_path,
        "paths_path": paths_path,
        "color_order": "RGB",
        "cache_source": "built_rgb64_v2",
    }

def resolve_heldout_store(
    dataset_name: str,
    spec: Dict,
) -> Dict:
    raw_dir = Path(spec["raw_dir"])
    cache_dir = Path(spec["cache_dir"])
    cache_name = str(spec["cache_name"])

    rgb_feature = cache_dir / f"{cache_name}_rgb64_v2_features.npy"
    rgb_label = cache_dir / f"{cache_name}_rgb64_v2_labels.npy"
    rgb_paths = cache_dir / f"{cache_name}_rgb64_v2_paths.json"

    legacy_feature = cache_dir / f"{cache_name}_features.npy"
    legacy_label = cache_dir / f"{cache_name}_labels.npy"
    legacy_paths = cache_dir / f"{cache_name}_paths.json"

    # Sama dengan training: RGB-v2 diprioritaskan; legacy dipakai jika v2 tidak ada.
    if rgb_feature.exists() and rgb_label.exists():
        print(f"✅ {dataset_name}: RGB-v2 cache")
        return {
            "feature_path": rgb_feature,
            "label_path": rgb_label,
            "paths_path": rgb_paths if rgb_paths.exists() else None,
            "color_order": "RGB",
            "cache_source": "rgb64_v2",
        }

    if (
        PREFER_LEGACY_CACHE
        and legacy_feature.exists()
        and legacy_label.exists()
    ):
        print(
            f"✅ {dataset_name}: legacy cache "
            f"({LEGACY_CACHE_COLOR_ORDER}→RGB saat loading)"
        )
        return {
            "feature_path": legacy_feature,
            "label_path": legacy_label,
            "paths_path": legacy_paths if legacy_paths.exists() else None,
            "color_order": LEGACY_CACHE_COLOR_ORDER,
            "cache_source": "legacy",
        }

    if not ALLOW_BUILD_CACHE:
        raise FileNotFoundError(
            f"Cache {dataset_name} tidak ada dan ALLOW_BUILD_CACHE=False."
        )

    if not raw_dir.exists():
        raise FileNotFoundError(
            f"Cache dan raw folder tidak ditemukan untuk {dataset_name}.\n"
            f"Raw expected: {raw_dir}"
        )

    print(
        f"ℹ️ {dataset_name}: cache belum ada, dibuat dari {raw_dir}"
    )

    return build_rgb_cache(
        raw_dir=raw_dir,
        cache_dir=cache_dir,
        cache_name=cache_name,
        nonviolence_dir=spec.get("nonviolence_dir"),
        violence_dir=spec.get("violence_dir"),
    )

## 4. Frame-diversity sampling pada fixed T=8 grid

Fixed raw-cache positions:

```text
[1,3,5,7,9,11,13,15]
```

Untuk K<8:
1. 8 segment positions dibagi menjadi K equal-width temporal bins;
2. satu position dipilih secara reproducible dari tiap bin;
3. selected positions tetap kronologis;
4. selected positions diulang secara equal-weight hingga sequence kembali T=8.

Contoh K=4:

```text
distinct segment positions : [0,2,5,7]
raw-cache positions        : [1,5,11,15]

expanded positions         : [0,0,2,2,5,5,7,7]
expanded raw-cache indices : [1,1,5,5,11,11,15,15]
```

Karena model memakai TSM, duplicate adjacency tersebut memang merupakan bagian
dari perturbasi yang dievaluasi.

In [ ]:
IMAGENET_MEAN = torch.tensor(
    [
        0.485,
        0.456,
        0.406,
    ],
    dtype=torch.float32,
).view(
    1,
    3,
    1,
    1,
)

IMAGENET_STD = torch.tensor(
    [
        0.229,
        0.224,
        0.225,
    ],
    dtype=torch.float32,
).view(
    1,
    3,
    1,
    1,
)


def stable_dataset_code(
    dataset_name: str,
) -> int:
    return int(
        zlib.crc32(
            dataset_name.encode(
                "utf-8"
            )
        )
        & 0xFFFFFFFF
    )


def stratified_unique_segment_positions(
    dataset_name: str,
    source_idx: int,
    k_unique: int,
    repeat_id: int,
) -> np.ndarray:
    k_unique = int(
        k_unique
    )

    if k_unique not in K_VALUES:
        raise ValueError(
            f"K tidak valid: {k_unique}"
        )

    if (
        k_unique
        == NUM_SEGMENTS
    ):
        return np.arange(
            NUM_SEGMENTS,
            dtype=np.int64,
        )

    bin_width = (
        NUM_SEGMENTS
        // k_unique
    )

    seed_sequence = np.random.SeedSequence(
        [
            int(
                SAMPLING_BASE_SEED
            ),
            int(
                repeat_id
            ),
            int(
                stable_dataset_code(
                    dataset_name
                )
            ),
            int(
                source_idx
            ),
            int(
                k_unique
            ),
        ]
    )

    rng = np.random.default_rng(
        seed_sequence
    )

    selected = []

    for bin_id in range(
        k_unique
    ):
        start = (
            bin_id
            * bin_width
        )

        stop = (
            start
            + bin_width
        )

        selected.append(
            int(
                rng.integers(
                    start,
                    stop,
                )
            )
        )

    selected = np.asarray(
        selected,
        dtype=np.int64,
    )

    if (
        len(
            np.unique(
                selected
            )
        )
        != k_unique
    ):
        raise RuntimeError(
            "Jumlah distinct positions tidak sama dengan K."
        )

    if not np.all(
        np.diff(
            selected
        )
        > 0
    ):
        raise RuntimeError(
            "Distinct positions harus tetap kronologis."
        )

    return selected


def expand_equal_weight_to_t8(
    unique_positions: np.ndarray,
) -> np.ndarray:
    unique_positions = np.asarray(
        unique_positions,
        dtype=np.int64,
    )

    repeat_factor = (
        NUM_SEGMENTS
        // len(
            unique_positions
        )
    )

    expanded = np.repeat(
        unique_positions,
        repeat_factor,
    ).astype(
        np.int64
    )

    if (
        len(
            expanded
        )
        != NUM_SEGMENTS
    ):
        raise RuntimeError(
            "Expanded sequence harus T=8."
        )

    counts = np.bincount(
        expanded,
        minlength=NUM_SEGMENTS,
    )

    nonzero_counts = counts[
        counts
        > 0
    ]

    if (
        len(
            np.unique(
                nonzero_counts
            )
        )
        != 1
    ):
        raise RuntimeError(
            "Equal-weight expansion gagal."
        )

    return expanded


# ------------------------------------------------------------
# Determinism / protocol audit
# ------------------------------------------------------------
audit_rows = []

for k_unique in K_VALUES:
    repeat_ids = (
        [0]
        if k_unique == 8
        else range(
            3
        )
    )

    for repeat_id in repeat_ids:
        first = (
            stratified_unique_segment_positions(
                dataset_name="RWF",
                source_idx=123,
                k_unique=k_unique,
                repeat_id=repeat_id,
            )
        )

        second = (
            stratified_unique_segment_positions(
                dataset_name="RWF",
                source_idx=123,
                k_unique=k_unique,
                repeat_id=repeat_id,
            )
        )

        if not np.array_equal(
            first,
            second,
        ):
            raise RuntimeError(
                "Frame sampling tidak deterministik."
            )

        expanded = (
            expand_equal_weight_to_t8(
                first
            )
        )

        audit_rows.append(
            {
                "k_unique": k_unique,
                "repeat": repeat_id,
                "distinct_segment_positions_0based": first.tolist(),
                "distinct_raw_cache_indices_0based": (
                    TEST_SEGMENT_INDICES[
                        first
                    ]
                ).tolist(),
                "expanded_segment_positions_0based": expanded.tolist(),
                "expanded_raw_cache_indices_0based": (
                    TEST_SEGMENT_INDICES[
                        expanded
                    ]
                ).tolist(),
            }
        )

frame_selection_audit_df = pd.DataFrame(
    audit_rows
)

frame_selection_audit_df.to_json(
    EVAL_OUTPUT_DIR
    / "frame_selection_audit_examples.json",
    orient="records",
    indent=2,
)

display(
    frame_selection_audit_df
)

print(
    "✅ Frame-selection generator audit PASS"
)

,k_unique,repeat,distinct_segment_positions_0based,distinct_raw_cache_indices_0based,expanded_segment_positions_0based,expanded_raw_cache_indices_0based
0,1,0,[7],[15],"[7, 7, 7, 7, 7, 7, 7, 7]","[15, 15, 15, 15, 15, 15, 15, 15]"
1,1,1,[4],[9],"[4, 4, 4, 4, 4, 4, 4, 4]","[9, 9, 9, 9, 9, 9, 9, 9]"
2,1,2,[0],[1],"[0, 0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1]"
3,2,0,"[3, 6]","[7, 13]","[3, 3, 3, 3, 6, 6, 6, 6]","[7, 7, 7, 7, 13, 13, 13, 13]"
4,2,1,"[2, 5]","[5, 11]","[2, 2, 2, 2, 5, 5, 5, 5]","[5, 5, 5, 5, 11, 11, 11, 11]"
5,2,2,"[2, 4]","[5, 9]","[2, 2, 2, 2, 4, 4, 4, 4]","[5, 5, 5, 5, 9, 9, 9, 9]"
6,4,0,"[0, 2, 5, 7]","[1, 5, 11, 15]","[0, 0, 2, 2, 5, 5, 7, 7]","[1, 1, 5, 5, 11, 11, 15, 15]"
7,4,1,"[0, 3, 5, 7]","[1, 7, 11, 15]","[0, 0, 3, 3, 5, 5, 7, 7]","[1, 1, 7, 7, 11, 11, 15, 15]"
8,4,2,"[1, 2, 4, 6]","[3, 5, 9, 13]","[1, 1, 2, 2, 4, 4, 6, 6]","[3, 3, 5, 5, 9, 9, 13, 13]"
9,8,0,"[0, 1, 2, 3, 4, 5, 6, 7]","[1, 3, 5, 7, 9, 11, 13, 15]","[0, 1, 2, 3, 4, 5, 6, 7]","[1, 3, 5, 7, 9, 11, 13, 15]"


✅ Frame-selection generator audit PASS


## 5. Direct-T8 dan Frame-Diversity datasets

Dua jalur digunakan:

- `DirectT8`: memakai `[1,3,5,7,9,11,13,15]` langsung;
- `FrameDiversity`: memakai K distinct positions lalu equal-weight repetition
  kembali ke T=8.

K=8 FrameDiversity harus mereproduksi DirectT8 sebelum K<8 dijalankan.

In [ ]:
class HeldOut16CacheBase(
    Dataset
):
    def __init__(
        self,
        dataset_name: str,
        feature_path: Path,
        label_path: Path,
        color_order: str,
    ):
        self.dataset_name = str(
            dataset_name
        )

        self.features = np.load(
            feature_path,
            mmap_mode="r",
        )

        self.labels = np.load(
            label_path,
            mmap_mode="r",
        ).astype(
            np.int64
        )

        self.color_order = str(
            color_order
        ).upper()

        if (
            len(
                self.features
            )
            != len(
                self.labels
            )
        ):
            raise ValueError(
                f"{dataset_name}: feature/label length mismatch."
            )

        expected_shape = (
            CACHE_SEQUENCE_LENGTH,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        )

        if (
            self.features.ndim
            != 5
            or tuple(
                self.features.shape[
                    1:
                ]
            )
            != expected_shape
        ):
            raise ValueError(
                f"{dataset_name}: cache shape={self.features.shape}; "
                f"expected [N,{CACHE_SEQUENCE_LENGTH},"
                f"{RAW_HEIGHT},{RAW_WIDTH},3]."
            )

        unique_labels = set(
            np.unique(
                self.labels
            ).tolist()
        )

        if (
            unique_labels
            != {
                0,
                1,
            }
        ):
            raise ValueError(
                f"{dataset_name}: held-out labels harus {{0,1}}, "
                f"found {sorted(unique_labels)}."
            )

        if self.color_order not in {
            "RGB",
            "BGR",
        }:
            raise ValueError(
                f"Unsupported color_order={self.color_order}"
            )

    def __len__(
        self,
    ):
        return len(
            self.labels
        )

    def load_frames(
        self,
        source_idx: int,
    ) -> Tuple[np.ndarray, int]:
        frames = np.asarray(
            self.features[
                source_idx
            ],
            dtype=np.float32,
        )

        label = int(
            self.labels[
                source_idx
            ]
        )

        if not np.isfinite(
            frames
        ).all():
            raise ValueError(
                f"{self.dataset_name}, idx={source_idx}: NaN/Inf."
            )

        frame_min = float(
            frames.min()
        )

        frame_max = float(
            frames.max()
        )

        if (
            AUTO_SCALE_LEGACY_CACHE
            and frame_min >= 0.0
            and frame_max > 1.5
        ):
            if (
                frame_max
                <= 255.0
                + 1e-3
            ):
                frames = (
                    frames
                    / 255.0
                )

            else:
                raise ValueError(
                    f"{self.dataset_name}: invalid cache range "
                    f"[{frame_min},{frame_max}]."
                )

        if (
            float(
                frames.min()
            )
            < -1e-4
            or float(
                frames.max()
            )
            > 1.0001
        ):
            raise ValueError(
                f"{self.dataset_name}: cache must be [0,1]."
            )

        if self.color_order == "BGR":
            frames = frames[
                ...,
                [
                    2,
                    1,
                    0,
                ],
            ]

        return (
            frames,
            label,
        )


def selected_to_tensor(
    selected: np.ndarray,
) -> torch.Tensor:
    if (
        tuple(
            selected.shape
        )
        != (
            NUM_SEGMENTS,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        )
    ):
        raise RuntimeError(
            f"Selected clip shape salah: {selected.shape}"
        )

    clip = torch.from_numpy(
        np.ascontiguousarray(
            selected
        )
    ).permute(
        0,
        3,
        1,
        2,
    )

    return (
        clip
        - IMAGENET_MEAN
    ) / IMAGENET_STD


class MobileNetTSMDirectT8Dataset(
    HeldOut16CacheBase
):
    def __getitem__(
        self,
        source_idx: int,
    ):
        (
            frames,
            label,
        ) = self.load_frames(
            source_idx
        )

        selected = frames[
            TEST_SEGMENT_INDICES
        ]

        clip = selected_to_tensor(
            selected
        )

        return (
            clip,
            torch.tensor(
                label,
                dtype=torch.long,
            ),
            int(
                source_idx
            ),
        )


class MobileNetTSMFrameDiversityDataset(
    HeldOut16CacheBase
):
    def __init__(
        self,
        dataset_name: str,
        feature_path: Path,
        label_path: Path,
        color_order: str,
        k_unique: int,
        repeat_id: int,
    ):
        super().__init__(
            dataset_name=dataset_name,
            feature_path=feature_path,
            label_path=label_path,
            color_order=color_order,
        )

        self.k_unique = int(
            k_unique
        )

        self.repeat_id = int(
            repeat_id
        )

    def __getitem__(
        self,
        source_idx: int,
    ):
        (
            frames,
            label,
        ) = self.load_frames(
            source_idx
        )

        selected_8 = frames[
            TEST_SEGMENT_INDICES
        ]

        unique_positions = (
            stratified_unique_segment_positions(
                dataset_name=self.dataset_name,
                source_idx=int(
                    source_idx
                ),
                k_unique=self.k_unique,
                repeat_id=self.repeat_id,
            )
        )

        expanded_positions = (
            expand_equal_weight_to_t8(
                unique_positions
            )
        )

        selected = selected_8[
            expanded_positions
        ]

        clip = selected_to_tensor(
            selected
        )

        return (
            clip,
            torch.tensor(
                label,
                dtype=torch.long,
            ),
            int(
                source_idx
            ),
            torch.from_numpy(
                unique_positions.copy()
            ),
            torch.from_numpy(
                expanded_positions.copy()
            ),
        )


def make_loader(
    dataset: Dataset,
):
    generator = torch.Generator()

    generator.manual_seed(
        SAMPLING_BASE_SEED
    )

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=(
            DEVICE.type
            == "cuda"
        ),
        worker_init_fn=seed_worker,
        generator=generator,
        persistent_workers=(
            NUM_WORKERS
            > 0
        ),
    )


def load_paths_if_available(
    paths_path: Optional[Path],
    expected_n: int,
) -> Optional[List[str]]:
    if paths_path is None:
        return None

    path = Path(
        paths_path
    )

    if not path.exists():
        return None

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as file:
        payload = json.load(
            file
        )

    paths = payload.get(
        "paths"
    )

    if not isinstance(
        paths,
        list,
    ):
        return None

    if (
        len(
            paths
        )
        != expected_n
    ):
        return None

    return [
        str(
            path
        )
        for path
        in paths
    ]

## 6. Resolve dan audit RWF, UCF, Bus Violence

In [ ]:
heldout_stores = {}

dataset_audit_rows = []

for (
    dataset_name,
    spec,
) in HELDOUT_SPECS.items():
    print(
        "\n"
        + "=" * 90
    )

    print(
        "DATASET:",
        dataset_name,
    )

    print(
        "=" * 90
    )

    store = resolve_heldout_store(
        dataset_name,
        spec,
    )

    dataset = MobileNetTSMDirectT8Dataset(
        dataset_name=dataset_name,
        feature_path=store[
            "feature_path"
        ],
        label_path=store[
            "label_path"
        ],
        color_order=store[
            "color_order"
        ],
    )

    labels = np.asarray(
        dataset.labels,
        dtype=np.int64,
    )

    counts = np.bincount(
        labels,
        minlength=2,
    )

    heldout_stores[
        dataset_name
    ] = store

    dataset_audit_rows.append(
        {
            "dataset": dataset_name,
            "n_total": int(
                len(
                    dataset
                )
            ),
            "n_nonviolence": int(
                counts[
                    0
                ]
            ),
            "n_violence": int(
                counts[
                    1
                ]
            ),
            "cache_source": store[
                "cache_source"
            ],
            "color_order": store[
                "color_order"
            ],
            "feature_path": str(
                store[
                    "feature_path"
                ]
            ),
            "label_path": str(
                store[
                    "label_path"
                ]
            ),
        }
    )

    print(
        f"✅ {dataset_name}: "
        f"N={len(dataset)}, "
        f"NV={counts[0]}, "
        f"V={counts[1]}, "
        f"cache={store['cache_source']}"
    )

dataset_audit_df = pd.DataFrame(
    dataset_audit_rows
)

dataset_audit_df.to_csv(
    EVAL_OUTPUT_DIR
    / "heldout_dataset_audit.csv",
    index=False,
)

display(
    dataset_audit_df
)


DATASET: RWF
✅ RWF: legacy cache (BGR→RGB saat loading)
✅ RWF: N=1998, NV=1000, V=998, cache=legacy

DATASET: UCF
✅ UCF: RGB-v2 cache
✅ UCF: N=100, NV=50, V=50, cache=rgb64_v2

DATASET: BV
✅ BV: RGB-v2 cache
✅ BV: N=1400, NV=700, V=700, cache=rgb64_v2


,dataset,n_total,n_nonviolence,n_violence,cache_source,color_order,feature_path,label_path
0,RWF,1998,1000,998,legacy,BGR,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...
1,UCF,100,50,50,rgb64_v2,RGB,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...
2,BV,1400,700,700,rgb64_v2,RGB,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...


## 7. Metrics dan per-clip prediction export

Setiap prediction row menyimpan:
- dataset;
- checkpoint seed;
- K;
- repeat;
- sample key;
- distinct positions;
- expanded T=8 positions;
- label;
- prediction;
- correctness;
- class probabilities.

Variabel prediction dataframe sengaja menggunakan nama yang eksplisit
(`direct_predictions_df`, `diversity_predictions_df`, `condition_predictions_df`)
untuk menghindari typo `prediction_df` yang sebelumnya menyebabkan `NameError`.

In [ ]:
def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    p_violence: np.ndarray,
) -> Dict[str, float]:
    (
        precision_cls,
        recall_cls,
        f1_cls,
        support,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[
            0,
            1,
        ],
        zero_division=0,
    )

    result = {
        "n": int(
            len(
                y_true
            )
        ),

        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "precision_macro": float(
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),

        "recall_macro": float(
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "f1_macro": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),

        "precision_nonviolence": float(
            precision_cls[
                0
            ]
        ),

        "recall_nonviolence": float(
            recall_cls[
                0
            ]
        ),

        "f1_nonviolence": float(
            f1_cls[
                0
            ]
        ),

        "precision_violence": float(
            precision_cls[
                1
            ]
        ),

        "recall_violence": float(
            recall_cls[
                1
            ]
        ),

        "f1_violence": float(
            f1_cls[
                1
            ]
        ),
    }

    if (
        len(
            np.unique(
                y_true
            )
        )
        == 2
    ):
        result[
            "roc_auc"
        ] = float(
            roc_auc_score(
                y_true,
                p_violence,
            )
        )

        result[
            "average_precision"
        ] = float(
            average_precision_score(
                y_true,
                p_violence,
            )
        )

    else:
        result[
            "roc_auc"
        ] = float(
            "nan"
        )

        result[
            "average_precision"
        ] = float(
            "nan"
        )

    return result


@torch.inference_mode()
def run_direct_loader(
    loader: DataLoader,
):
    model.eval()

    all_indices = []
    all_labels = []
    all_predictions = []
    all_probabilities = []

    for (
        clips,
        labels,
        indices,
    ) in tqdm(
        loader,
        leave=False,
        desc="Direct T8",
    ):
        clips = clips.to(
            DEVICE,
            non_blocking=True,
        )

        logits = model(
            clips
        )

        assert_finite_tensor(
            "direct_logits",
            logits,
        )

        probabilities = torch.softmax(
            logits.float(),
            dim=1,
        )

        predictions = torch.argmax(
            probabilities,
            dim=1,
        )

        all_indices.extend(
            np.asarray(
                indices
            ).tolist()
        )

        all_labels.extend(
            labels.numpy().tolist()
        )

        all_predictions.extend(
            predictions.cpu().numpy().tolist()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy().tolist()
        )

    return (
        np.asarray(
            all_indices,
            dtype=np.int64,
        ),
        np.asarray(
            all_labels,
            dtype=np.int64,
        ),
        np.asarray(
            all_predictions,
            dtype=np.int64,
        ),
        np.asarray(
            all_probabilities,
            dtype=np.float64,
        ),
    )


@torch.inference_mode()
def run_diversity_loader(
    loader: DataLoader,
):
    model.eval()

    all_indices = []
    all_labels = []
    all_predictions = []
    all_probabilities = []

    all_unique_positions = []
    all_expanded_positions = []

    for (
        clips,
        labels,
        indices,
        unique_positions,
        expanded_positions,
    ) in tqdm(
        loader,
        leave=False,
        desc="Frame diversity",
    ):
        clips = clips.to(
            DEVICE,
            non_blocking=True,
        )

        logits = model(
            clips
        )

        assert_finite_tensor(
            "diversity_logits",
            logits,
        )

        probabilities = torch.softmax(
            logits.float(),
            dim=1,
        )

        predictions = torch.argmax(
            probabilities,
            dim=1,
        )

        all_indices.extend(
            np.asarray(
                indices
            ).tolist()
        )

        all_labels.extend(
            labels.numpy().tolist()
        )

        all_predictions.extend(
            predictions.cpu().numpy().tolist()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy().tolist()
        )

        all_unique_positions.extend(
            unique_positions.numpy().tolist()
        )

        all_expanded_positions.extend(
            expanded_positions.numpy().tolist()
        )

    return (
        np.asarray(
            all_indices,
            dtype=np.int64,
        ),
        np.asarray(
            all_labels,
            dtype=np.int64,
        ),
        np.asarray(
            all_predictions,
            dtype=np.int64,
        ),
        np.asarray(
            all_probabilities,
            dtype=np.float64,
        ),
        all_unique_positions,
        all_expanded_positions,
    )


def resolve_sample_keys(
    dataset_name: str,
    store: Dict,
    indices: np.ndarray,
):
    source_paths = load_paths_if_available(
        store.get(
            "paths_path"
        ),
        expected_n=len(
            indices
        ),
    )

    if source_paths is None:
        sample_keys = [
            f"{dataset_name}::{int(index)}"
            for index
            in indices
        ]

        video_paths = [
            ""
            for _
            in indices
        ]

    else:
        video_paths = [
            source_paths[
                int(
                    index
                )
            ]
            for index
            in indices
        ]

        sample_keys = [
            Path(
                path
            ).as_posix()
            for path
            in video_paths
        ]

    return (
        sample_keys,
        video_paths,
    )


def evaluate_direct_condition(
    dataset_name: str,
    store: Dict,
):
    dataset = MobileNetTSMDirectT8Dataset(
        dataset_name=dataset_name,
        feature_path=store[
            "feature_path"
        ],
        label_path=store[
            "label_path"
        ],
        color_order=store[
            "color_order"
        ],
    )

    loader = make_loader(
        dataset
    )

    (
        indices,
        y_true,
        y_pred,
        probabilities,
    ) = run_direct_loader(
        loader
    )

    (
        sample_keys,
        video_paths,
    ) = resolve_sample_keys(
        dataset_name,
        store,
        indices,
    )

    metrics = compute_metrics(
        y_true,
        y_pred,
        probabilities[
            :,
            1
        ],
    )

    direct_predictions_df = pd.DataFrame(
        {
            "checkpoint_seed": RUN_SEED,
            "dataset": dataset_name,
            "condition": "direct_t8",
            "k_unique": 8,
            "repeat": 0,
            "sample_index": indices,
            "sample_key": sample_keys,
            "video_path": video_paths,
            "y_true": y_true,
            "y_pred": y_pred,
            "is_correct": (
                y_true
                == y_pred
            ).astype(
                np.int64
            ),
            "p_nonviolence": probabilities[
                :,
                0
            ],
            "p_violence": probabilities[
                :,
                1
            ],
        }
    )

    if direct_predictions_df[
        "sample_key"
    ].duplicated().any():
        raise RuntimeError(
            f"Duplicate sample_key: {dataset_name} direct."
        )

    return (
        metrics,
        direct_predictions_df,
    )


def evaluate_diversity_condition(
    dataset_name: str,
    store: Dict,
    k_unique: int,
    repeat_id: int,
):
    dataset = MobileNetTSMFrameDiversityDataset(
        dataset_name=dataset_name,
        feature_path=store[
            "feature_path"
        ],
        label_path=store[
            "label_path"
        ],
        color_order=store[
            "color_order"
        ],
        k_unique=k_unique,
        repeat_id=repeat_id,
    )

    loader = make_loader(
        dataset
    )

    (
        indices,
        y_true,
        y_pred,
        probabilities,
        unique_positions,
        expanded_positions,
    ) = run_diversity_loader(
        loader
    )

    (
        sample_keys,
        video_paths,
    ) = resolve_sample_keys(
        dataset_name,
        store,
        indices,
    )

    unique_segment_strings = [
        json.dumps(
            positions
        )
        for positions
        in unique_positions
    ]

    expanded_segment_strings = [
        json.dumps(
            positions
        )
        for positions
        in expanded_positions
    ]

    unique_raw_strings = [
        json.dumps(
            TEST_SEGMENT_INDICES[
                np.asarray(
                    positions,
                    dtype=np.int64,
                )
            ].tolist()
        )
        for positions
        in unique_positions
    ]

    expanded_raw_strings = [
        json.dumps(
            TEST_SEGMENT_INDICES[
                np.asarray(
                    positions,
                    dtype=np.int64,
                )
            ].tolist()
        )
        for positions
        in expanded_positions
    ]

    metrics = compute_metrics(
        y_true,
        y_pred,
        probabilities[
            :,
            1
        ],
    )

    diversity_predictions_df = pd.DataFrame(
        {
            "checkpoint_seed": RUN_SEED,
            "dataset": dataset_name,
            "condition": "frame_diversity",
            "k_unique": k_unique,
            "repeat": repeat_id,
            "sample_index": indices,
            "sample_key": sample_keys,
            "video_path": video_paths,
            "distinct_segment_positions_0based": unique_segment_strings,
            "distinct_raw_cache_indices_0based": unique_raw_strings,
            "expanded_segment_positions_0based": expanded_segment_strings,
            "expanded_raw_cache_indices_0based": expanded_raw_strings,
            "y_true": y_true,
            "y_pred": y_pred,
            "is_correct": (
                y_true
                == y_pred
            ).astype(
                np.int64
            ),
            "p_nonviolence": probabilities[
                :,
                0
            ],
            "p_violence": probabilities[
                :,
                1
            ],
        }
    )

    if diversity_predictions_df[
        "sample_key"
    ].duplicated().any():
        raise RuntimeError(
            f"Duplicate sample_key: "
            f"{dataset_name}, K={k_unique}, repeat={repeat_id}."
        )

    return (
        metrics,
        diversity_predictions_df,
    )

## 8. K=8 reproduction gate

Sebelum K=1/2/4 dijalankan:

```text
Direct T=8
vs
FrameDiversity K=8
```

harus sama clip-by-clip.

Gate memeriksa:
- jumlah sample;
- `sample_key`;
- label;
- `p_nonviolence`;
- `p_violence`;
- Accuracy;
- Balanced Accuracy;
- Macro-F1;
- ROC-AUC;
- AP.

Jika gate gagal, notebook berhenti.

In [ ]:
direct_metric_rows = []

direct_prediction_frames = []

k8_metric_rows = []

k8_prediction_frames = []

gate_rows = []

for (
    dataset_name,
    store,
) in heldout_stores.items():
    print(
        "\n"
        + "=" * 90
    )

    print(
        f"K=8 REPRODUCTION GATE | {dataset_name}"
    )

    print(
        "=" * 90
    )

    (
        direct_metrics,
        direct_predictions_df,
    ) = evaluate_direct_condition(
        dataset_name,
        store,
    )

    (
        k8_metrics,
        k8_predictions_df,
    ) = evaluate_diversity_condition(
        dataset_name=dataset_name,
        store=store,
        k_unique=8,
        repeat_id=0,
    )

    merged = direct_predictions_df[
        [
            "sample_key",
            "y_true",
            "p_nonviolence",
            "p_violence",
        ]
    ].merge(
        k8_predictions_df[
            [
                "sample_key",
                "y_true",
                "p_nonviolence",
                "p_violence",
            ]
        ],
        on="sample_key",
        how="inner",
        suffixes=(
            "_direct",
            "_k8",
        ),
        validate="one_to_one",
    )

    if (
        len(
            merged
        )
        != len(
            direct_predictions_df
        )
    ):
        raise RuntimeError(
            f"{dataset_name}: direct/K8 sample-count mismatch."
        )

    if not np.array_equal(
        merged[
            "y_true_direct"
        ].to_numpy(),
        merged[
            "y_true_k8"
        ].to_numpy(),
    ):
        raise RuntimeError(
            f"{dataset_name}: direct/K8 label mismatch."
        )

    max_probability_delta = float(
        np.max(
            np.abs(
                merged[
                    [
                        "p_nonviolence_direct",
                        "p_violence_direct",
                    ]
                ].to_numpy()
                -
                merged[
                    [
                        "p_nonviolence_k8",
                        "p_violence_k8",
                    ]
                ].to_numpy()
            )
        )
    )

    metric_deltas = {}

    for metric_name in [
        "accuracy",
        "balanced_accuracy",
        "f1_macro",
        "roc_auc",
        "average_precision",
    ]:
        metric_deltas[
            metric_name
        ] = float(
            k8_metrics[
                metric_name
            ]
            -
            direct_metrics[
                metric_name
            ]
        )

    finite_metric_deltas = [
        abs(
            value
        )
        for value
        in metric_deltas.values()
        if np.isfinite(
            value
        )
    ]

    max_metric_delta = (
        float(
            max(
                finite_metric_deltas
            )
        )
        if finite_metric_deltas
        else 0.0
    )

    gate_rows.append(
        {
            "checkpoint_seed": RUN_SEED,
            "dataset": dataset_name,
            "max_abs_probability_delta": max_probability_delta,
            "max_abs_metric_delta": max_metric_delta,
            **{
                f"delta_{key}": value
                for key, value
                in metric_deltas.items()
            },
        }
    )

    if (
        max_probability_delta
        > K8_PROBABILITY_ATOL
    ):
        raise RuntimeError(
            f"{dataset_name}: K8 probability reproduction failed: "
            f"{max_probability_delta} > {K8_PROBABILITY_ATOL}"
        )

    if (
        max_metric_delta
        > K8_METRIC_ATOL
    ):
        raise RuntimeError(
            f"{dataset_name}: K8 metric reproduction failed: "
            f"{max_metric_delta} > {K8_METRIC_ATOL}"
        )

    direct_metric_rows.append(
        {
            "checkpoint_seed": RUN_SEED,
            "dataset": dataset_name,
            **direct_metrics,
        }
    )

    direct_prediction_frames.append(
        direct_predictions_df
    )

    k8_metric_rows.append(
        {
            "checkpoint_seed": RUN_SEED,
            "k_unique": 8,
            "repeat": 0,
            "dataset": dataset_name,
            **k8_metrics,
        }
    )

    k8_prediction_frames.append(
        k8_predictions_df
    )

    print(
        f"✅ {dataset_name}: "
        f"max Δprob={max_probability_delta:.3e}, "
        f"max Δmetric={max_metric_delta:.3e}"
    )

gate_df = pd.DataFrame(
    gate_rows
)

gate_df.to_csv(
    EVAL_OUTPUT_DIR
    / f"mobilenettsm_K8_direct_reproduction_gate_seed{RUN_SEED}.csv",
    index=False,
)

pd.DataFrame(
    direct_metric_rows
).to_csv(
    EVAL_OUTPUT_DIR
    / f"mobilenettsm_direct_T8_metrics_seed{RUN_SEED}.csv",
    index=False,
)

pd.concat(
    direct_prediction_frames,
    ignore_index=True,
).to_csv(
    EVAL_OUTPUT_DIR
    / f"mobilenettsm_direct_T8_predictions_seed{RUN_SEED}.csv",
    index=False,
)

display(
    gate_df
)

print(
    "\n✅ Semua K=8 reproduction gates PASS"
)


K=8 REPRODUCTION GATE | RWF


Direct T8:   0%|          | 0/250 [00:00<?, ?it/s]

Frame diversity:   0%|          | 0/250 [00:00<?, ?it/s]

✅ RWF: max Δprob=0.000e+00, max Δmetric=0.000e+00

K=8 REPRODUCTION GATE | UCF


Direct T8:   0%|          | 0/13 [00:00<?, ?it/s]

...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d6f39e831a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d6f39e831a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Frame diversity:   0%|          | 0/175 [00:00<?, ?it/s]

✅ BV: max Δprob=0.000e+00, max Δmetric=0.000e+00


,checkpoint_seed,dataset,max_abs_probability_delta,max_abs_metric_delta,delta_accuracy,delta_balanced_accuracy,delta_f1_macro,delta_roc_auc,delta_average_precision
0,0,RWF,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,UCF,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,BV,0.0,0.0,0.0,0.0,0.0,0.0,0.0



✅ Semua K=8 reproduction gates PASS


## 9. Jalankan K=1/2/4 × 10 repeats + K=8 reference

In [ ]:
metric_frames = [
    pd.DataFrame(
        k8_metric_rows
    )
]

prediction_frames = [
    pd.concat(
        k8_prediction_frames,
        ignore_index=True,
    )
]

for k_unique in [
    1,
    2,
    4,
]:
    for repeat_id in range(
        NUM_RANDOM_REPEATS
    ):
        print(
            f"\nseed={RUN_SEED} | "
            f"K={k_unique} | "
            f"repeat={repeat_id}"
        )

        condition_metric_rows = []

        condition_prediction_frames = []

        for (
            dataset_name,
            store,
        ) in heldout_stores.items():
            (
                metrics,
                condition_predictions_df,
            ) = evaluate_diversity_condition(
                dataset_name=dataset_name,
                store=store,
                k_unique=k_unique,
                repeat_id=repeat_id,
            )

            condition_metric_rows.append(
                {
                    "checkpoint_seed": RUN_SEED,
                    "k_unique": k_unique,
                    "repeat": repeat_id,
                    "dataset": dataset_name,
                    **metrics,
                }
            )

            condition_prediction_frames.append(
                condition_predictions_df
            )

        metric_frames.append(
            pd.DataFrame(
                condition_metric_rows
            )
        )

        prediction_frames.append(
            pd.concat(
                condition_prediction_frames,
                ignore_index=True,
            )
        )

all_metrics = pd.concat(
    metric_frames,
    ignore_index=True,
)

all_predictions = pd.concat(
    prediction_frames,
    ignore_index=True,
)

metrics_path = (
    EVAL_OUTPUT_DIR
    / f"mobilenettsm_frame_diversity_all_repeats_seed{RUN_SEED}.csv"
)

predictions_path = (
    EVAL_OUTPUT_DIR
    / f"mobilenettsm_frame_diversity_predictions_seed{RUN_SEED}.csv"
)

all_metrics.to_csv(
    metrics_path,
    index=False,
)

all_predictions.to_csv(
    predictions_path,
    index=False,
)

print(
    "\n✅ Semua held-out MobileNetV2+TSM frame-diversity conditions selesai."
)

print(
    "Metrics    :",
    metrics_path,
)

print(
    "Predictions:",
    predictions_path,
)


seed=0 | K=1 | repeat=0


Frame diversity:   0%|          | 0/250 [00:00<?, ?it/s]

Frame diversity:   0%|          | 0/13 [00:00<?, ?it/s]

Frame diversity:   0%|          | 0/175 [00:00<?, ?it/s]


seed=0 | K=1 | repeat=1


...


seed=0 | K=4 | repeat=9


Frame diversity:   0%|          | 0/250 [00:00<?, ?it/s]

Frame diversity:   0%|          | 0/13 [00:00<?, ?it/s]

Frame diversity:   0%|          | 0/175 [00:00<?, ?it/s]


✅ Semua held-out MobileNetV2+TSM frame-diversity conditions selesai.
Metrics    : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/MobileNetV2_TSM_FrameDiversity/seed_0/mobilenettsm_frame_diversity_all_repeats_seed0.csv
Predictions: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/MobileNetV2_TSM_FrameDiversity/seed_0/mobilenettsm_frame_diversity_predictions_seed0.csv


## 10. Summary mean ± temporal-sampling std

Untuk K=1/2/4, `std_sampling` berasal dari **10 temporal-sampling repeats**,
bukan dari checkpoint seeds.

K=8 reference hanya satu deterministic condition dan memiliki
`std_sampling = 0`.

In [ ]:
METRIC_COLUMNS = [
    "accuracy",
    "precision_macro",
    "recall_macro",
    "balanced_accuracy",
    "f1_macro",
    "roc_auc",
    "average_precision",
    "precision_nonviolence",
    "recall_nonviolence",
    "f1_nonviolence",
    "precision_violence",
    "recall_violence",
    "f1_violence",
]

summary_rows = []

for (
    dataset_name,
    k_unique,
), group in all_metrics.groupby(
    [
        "dataset",
        "k_unique",
    ]
):
    row = {
        "checkpoint_seed": RUN_SEED,
        "dataset": dataset_name,
        "k_unique": int(
            k_unique
        ),
        "num_sampling_repeats": int(
            len(
                group
            )
        ),
    }

    for metric_name in METRIC_COLUMNS:
        values = group[
            metric_name
        ].astype(
            float
        )

        row[
            f"{metric_name}_mean"
        ] = float(
            values.mean()
        )

        row[
            f"{metric_name}_std_sampling"
        ] = float(
            values.std(
                ddof=1
            )
            if len(
                values
            )
            > 1
            else 0.0
        )

        row[
            f"{metric_name}_min"
        ] = float(
            values.min()
        )

        row[
            f"{metric_name}_max"
        ] = float(
            values.max()
        )

    summary_rows.append(
        row
    )

summary_df = pd.DataFrame(
    summary_rows
).sort_values(
    [
        "dataset",
        "k_unique",
    ]
)

summary_path = (
    EVAL_OUTPUT_DIR
    / f"mobilenettsm_frame_diversity_summary_seed{RUN_SEED}.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
)

display(
    summary_df[
        [
            "checkpoint_seed",
            "dataset",
            "k_unique",
            "num_sampling_repeats",
            "average_precision_mean",
            "average_precision_std_sampling",
            "roc_auc_mean",
            "roc_auc_std_sampling",
            "f1_macro_mean",
            "f1_macro_std_sampling",
        ]
    ].round(
        5
    )
)

print(
    "Saved:",
    summary_path,
)

,checkpoint_seed,dataset,k_unique,num_sampling_repeats,average_precision_mean,average_precision_std_sampling,roc_auc_mean,roc_auc_std_sampling,f1_macro_mean,f1_macro_std_sampling
0,0,BV,1,10,0.56379,0.00775,0.57303,0.00775,0.36485,0.00402
1,0,BV,2,10,0.57474,0.00563,0.59063,0.00677,0.35498,0.00346
2,0,BV,4,10,0.60839,0.00437,0.62327,0.00290,0.37561,0.00502
3,0,BV,8,1,0.65323,0.00000,0.66423,0.00000,0.43304,0.00000
4,0,RWF,1,10,0.55252,0.00657,0.55754,0.00470,0.47898,0.00662
5,0,RWF,2,10,0.58131,0.00420,0.59789,0.00322,0.51439,0.00660
6,0,RWF,4,10,0.64068,0.00174,0.65482,0.00388,0.58986,0.00560
7,0,RWF,8,1,0.72905,0.00000,0.72274,0.00000,0.66892,0.00000
8,0,UCF,1,10,0.54958,0.02107,0.55142,0.02910,0.48487,0.03559
9,0,UCF,2,10,0.58023,0.02447,0.58852,0.02090,0.50848,0.03567


Saved: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/MobileNetV2_TSM_FrameDiversity/seed_0/mobilenettsm_frame_diversity_summary_seed0.csv


## 11. Delta terhadap K=8

Primary paper metric: AP.

```text
ΔAP      = AP(K=8) - AP(K)
ΔAUC     = AUC(K=8) - AUC(K)
ΔMacroF1 = MacroF1(K=8) - MacroF1(K)
```

Kolom `delta_*_pp` sudah dalam **poin persentase**.

In [ ]:
k8_reference = (
    summary_df[
        summary_df[
            "k_unique"
        ]
        == 8
    ][
        [
            "dataset",
            "average_precision_mean",
            "roc_auc_mean",
            "f1_macro_mean",
        ]
    ]
    .rename(
        columns={
            "average_precision_mean": "ap_k8",
            "roc_auc_mean": "auc_k8",
            "f1_macro_mean": "f1_k8",
        }
    )
)

delta_df = summary_df.merge(
    k8_reference,
    on="dataset",
    how="left",
)

delta_df[
    "delta_ap_k8_minus_k"
] = (
    delta_df[
        "ap_k8"
    ]
    - delta_df[
        "average_precision_mean"
    ]
)

delta_df[
    "delta_auc_k8_minus_k"
] = (
    delta_df[
        "auc_k8"
    ]
    - delta_df[
        "roc_auc_mean"
    ]
)

delta_df[
    "delta_f1_k8_minus_k"
] = (
    delta_df[
        "f1_k8"
    ]
    - delta_df[
        "f1_macro_mean"
    ]
)

delta_df[
    "delta_ap_pp"
] = (
    100.0
    * delta_df[
        "delta_ap_k8_minus_k"
    ]
)

delta_df[
    "delta_auc_pp"
] = (
    100.0
    * delta_df[
        "delta_auc_k8_minus_k"
    ]
)

delta_df[
    "delta_f1_pp"
] = (
    100.0
    * delta_df[
        "delta_f1_k8_minus_k"
    ]
)

delta_path = (
    EVAL_OUTPUT_DIR
    / f"mobilenettsm_frame_diversity_delta_seed{RUN_SEED}.csv"
)

delta_df.to_csv(
    delta_path,
    index=False,
)

display(
    delta_df[
        [
            "dataset",
            "k_unique",
            "average_precision_mean",
            "delta_ap_pp",
            "roc_auc_mean",
            "delta_auc_pp",
            "f1_macro_mean",
            "delta_f1_pp",
        ]
    ].round(
        4
    )
)

print(
    "Saved:",
    delta_path,
)

,dataset,k_unique,average_precision_mean,delta_ap_pp,roc_auc_mean,delta_auc_pp,f1_macro_mean,delta_f1_pp
0,BV,1,0.5638,8.9441,0.5730,9.1199,0.3649,6.8189
1,BV,2,0.5747,7.8492,0.5906,7.3597,0.3550,7.8064
2,BV,4,0.6084,4.4842,0.6233,4.0964,0.3756,5.7438
3,BV,8,0.6532,0.0000,0.6642,0.0000,0.4330,0.0000
4,RWF,1,0.5525,17.6527,0.5575,16.5191,0.4790,18.9936
5,RWF,2,0.5813,14.7737,0.5979,12.4847,0.5144,15.4529
6,RWF,4,0.6407,8.8373,0.6548,6.7915,0.5899,7.9055
7,RWF,8,0.7291,0.0000,0.7227,0.0000,0.6689,0.0000
8,UCF,1,0.5496,15.3055,0.5514,16.7180,0.4849,19.3073
9,UCF,2,0.5802,12.2414,0.5885,13.0080,0.5085,16.9459


Saved: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/MobileNetV2_TSM_FrameDiversity/seed_0/mobilenettsm_frame_diversity_delta_seed0.csv


## 12. Protocol manifest

In [ ]:
protocol_audit = {
    "task": "heldout_frame_diversity_evaluation_only",
    "model": "Zhang MobileNetV2+TSM",
    "model_variant": MODEL_VARIANT,
    "checkpoint_seed": RUN_SEED,
    "checkpoint_path": str(
        CHECKPOINT_PATH
    ),
    "checkpoint_epoch": checkpoint.get(
        "epoch"
    ),
    "training_performed": False,
    "fine_tuning_performed": False,
    "strict_checkpoint_loading": True,
    "use_tsm": True,
    "tsm_method": (
        "Zhang Method-2: TemporalShift inserted after expansion "
        "in eligible residual InvertedResidual blocks"
    ),
    "tsm_blocks": int(
        model.tsm_blocks
    ),
    "shift_div": SHIFT_DIV,
    "cache_sequence_length": CACHE_SEQUENCE_LENGTH,
    "fixed_model_sequence_length": NUM_SEGMENTS,
    "fixed_test_segment_indices": (
        TEST_SEGMENT_INDICES
        .tolist()
    ),
    "tested_distinct_segment_counts": K_VALUES,
    "random_repeats_for_k_1_2_4": NUM_RANDOM_REPEATS,
    "sampling_base_seed": SAMPLING_BASE_SEED,
    "checkpoint_seed_excluded_from_sampling_generation": True,
    "sampling_identity_inputs": [
        "sampling_base_seed",
        "repeat_id",
        "dataset_name_crc32",
        "sample_index",
        "k_unique",
    ],
    "sampling_strategy": (
        "one reproducible random segment position from each "
        "equal-width temporal bin on the fixed T=8 grid"
    ),
    "expansion_strategy": (
        "selected distinct temporal positions repeated in chronological "
        "blocks with equal weight until T=8"
    ),
    "aggregation": "mean_of_per_segment_logits",
    "k8_direct_reproduction_gate": True,
    "k8_probability_atol": K8_PROBABILITY_ATOL,
    "k8_metric_atol": K8_METRIC_ATOL,
    "heldout_datasets": list(
        HELDOUT_SPECS.keys()
    ),
    "no_cross_dataset_pooling": True,
    "primary_metric": "average_precision",
    "important_k1_note": (
        "K=1 is not native-T=1 training. "
        "Checkpoint and model interface remain T=8."
    ),
    "interpretation_limit": (
        "Inference-time sensitivity to temporal diversity/redundancy "
        "on fixed T=8 MobileNetV2+TSM input. "
        "Because TSM exchanges features between neighboring positions, "
        "reducing K also changes local adjacency/redundancy. "
        "This is not a pure causal isolation and is not a compute-saving test."
    ),
}

manifest_path = (
    EVAL_OUTPUT_DIR
    / f"mobilenettsm_frame_diversity_protocol_seed{RUN_SEED}.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        protocol_audit,
        file,
        indent=2,
    )

print(
    json.dumps(
        protocol_audit,
        indent=2,
    )
)

print(
    "\nOutput:",
    EVAL_OUTPUT_DIR,
)

{
  "task": "heldout_frame_diversity_evaluation_only",
  "model": "Zhang MobileNetV2+TSM",
  "model_variant": "mobilenet_tsm",
  "checkpoint_seed": 0,
  "checkpoint_path": "/content/drive/MyDrive/IVS/ViolenceDetection/Models/Comparators/mobilenet_tsm_Seed0_v1/mobilenet_tsm/seed_0/best.pt",
  "checkpoint_epoch": 63,
  "training_performed": false,
  "fine_tuning_performed": false,
  "strict_checkpoint_loading": true,
  "use_tsm": true,
  "tsm_method": "Zhang Method-2: TemporalShift inserted after expansion in eligible residual InvertedResidual blocks",
  "tsm_blocks": 10,
  "shift_div": 8,
  "cache_sequence_length": 16,
  "fixed_model_sequence_length": 8,
  "fixed_test_segment_indices": [
    1,
    3,
    5,
    7,
    9,
    11,
    13,
    15
  ],
  "tested_distinct_segment_counts": [
    1,
    2,
    4,
    8
  ],
  "random_repeats_for_k_1_2_4": 10,
  "sampling_base_seed": 20260810,
  "checkpoint_seed_excluded_from_sampling_generation": true,
  "sampling_identity_inputs": [
    "sam

# Seed berikutnya

Untuk checkpoint berikutnya, **cukup ubah**:

```python
RUN_SEED = 0
```

menjadi:

```python
RUN_SEED = 42
```

kemudian:

```python
RUN_SEED = 123
```

Jangan ubah:

```python
NUM_SEGMENTS = 8
SHIFT_DIV = 8
TEST_SEGMENT_INDICES = [1,3,5,7,9,11,13,15]
K_VALUES = [1,2,4,8]
NUM_RANDOM_REPEATS = 10
SAMPLING_BASE_SEED = 20260810
```

Output otomatis dipisahkan:

```text
HeldOut_Evaluation/
└── MobileNetV2_TSM_FrameDiversity/
    ├── seed_0/
    ├── seed_42/
    └── seed_123/
```

File utama:
- `mobilenettsm_frame_diversity_all_repeats_seedX.csv`
- `mobilenettsm_frame_diversity_predictions_seedX.csv`
- `mobilenettsm_frame_diversity_summary_seedX.csv`
- `mobilenettsm_frame_diversity_delta_seedX.csv`
- `mobilenettsm_K8_direct_reproduction_gate_seedX.csv`
- `mobilenettsm_direct_T8_metrics_seedX.csv`
- `mobilenettsm_direct_T8_predictions_seedX.csv`
- `heldout_dataset_audit.csv`
- `frame_selection_audit_examples.json`
- `mobilenettsm_frame_diversity_protocol_seedX.json`